# Synthetic population — S1 verification

This notebook runs `synthesize_population()` and lets you inspect the output
visually before adding LLM enrichment (S2). No API key is needed — everything
here is pure random sampling.

## 1. Imports

In [ ]:
from collections import Counter

from aibm import ZoneSpec, synthesize_population

## 2. Define zone specs

Three zones with different demographic profiles:
- **Suburban** — families, cars, moderate income
- **Urban** — small households, fewer cars, mixed income
- **Student** — young adults, low income, few cars

In [ ]:
suburban = ZoneSpec(
    zone_id="suburban",
    n_households=80,
    household_size_dist={1: 0.1, 2: 0.3, 3: 0.35, 4: 0.25},
    age_dist={"0-17": 0.25, "18-64": 0.65, "65+": 0.10},
    employment_rate=0.70,
    student_rate=0.05,
    vehicle_dist={0: 0.1, 1: 0.5, 2: 0.4},
    income_dist={"low": 0.2, "medium": 0.55, "high": 0.25},
    license_rate=0.85,
)

urban = ZoneSpec(
    zone_id="urban",
    n_households=120,
    household_size_dist={1: 0.45, 2: 0.35, 3: 0.15, 4: 0.05},
    age_dist={"0-17": 0.12, "18-64": 0.70, "65+": 0.18},
    employment_rate=0.65,
    student_rate=0.10,
    vehicle_dist={0: 0.5, 1: 0.4, 2: 0.1},
    income_dist={"low": 0.3, "medium": 0.45, "high": 0.25},
    license_rate=0.70,
)

student_area = ZoneSpec(
    zone_id="student",
    n_households=50,
    household_size_dist={1: 0.3, 2: 0.4, 3: 0.2, 4: 0.1},
    age_dist={"0-17": 0.02, "18-64": 0.95, "65+": 0.03},
    employment_rate=0.30,
    student_rate=0.55,
    vehicle_dist={0: 0.65, 1: 0.30, 2: 0.05},
    income_dist={"low": 0.60, "medium": 0.35, "high": 0.05},
    license_rate=0.55,
)

specs = [suburban, urban, student_area]

## 3. Run synthesis

In [ ]:
households = synthesize_population(specs, seed=42)

all_agents = [agent for hh in households for agent in hh.members]

print(f"Households generated : {len(households)}")
print(f"Agents generated     : {len(all_agents)}")

## 4. Inspect home_zone propagation

In [ ]:
zone_counts = Counter(agent.home_zone for agent in all_agents)
print("Agents per zone:")
for zone, count in sorted(zone_counts.items()):
    print(f"  {zone}: {count}")

# Spot-check: each agent's home_zone matches their household's
mismatches = [
    (hh.home_zone, agent.home_zone)
    for hh in households
    for agent in hh.members
    if agent.home_zone != hh.home_zone
]
print(f"\nHome-zone mismatches: {len(mismatches)} (should be 0)")

## 5. Check distributions

In [ ]:
def bracket(age: int) -> str:
    if age < 18:
        return "0-17"
    if age <= 64:
        return "18-64"
    return "65+"


employment_counts = Counter(a.employment for a in all_agents)
age_bracket_counts = Counter(bracket(a.age) for a in all_agents)
license_count = sum(1 for a in all_agents if a.has_license)

n = len(all_agents)
print("Employment distribution:")
for status, count in sorted(employment_counts.items()):
    print(f"  {status:12s}: {count:4d}  ({count / n:.1%})")

print("\nAge-bracket distribution:")
for b, count in sorted(age_bracket_counts.items()):
    print(f"  {b:6s}: {count:4d}  ({count / n:.1%})")

print(f"\nAgents with driving licence: {license_count} ({license_count / n:.1%})")

## 6. Explore individual agents

In [ ]:
for hh in households[:5]:
    hh_summary = (
        f"{hh}  home_zone={hh.home_zone}"
        f"  vehicles={hh.num_vehicles}  income={hh.income_level}"
    )
    print(hh_summary)
    for agent in hh.members:
        license_str = "licence" if agent.has_license else "no licence"
        print(
            f"  {agent.name:12s}  age={agent.age:3d}"
            f"  {agent.employment:12s}  {license_str}"
        )
    print()